In [1]:

import numpy as np
from IPython.display import IFrame

In [2]:
import requests

In [3]:
# Start the Visor server on the command line using:
# visor-cli server start

# Take note of the service endpoint and assign to the url variable here.
url = "http://localhost:53211"

# Set up endpoints for the operations we will need
# Start
start_url = f"{url}/start"
# List datasets
list_datasets_url = f"{url}/list_datasets"
# List variables (for dataset)
def list_variables_url(dataset_id):
    return f"{url}/{dataset_id}/list_variables"
# Update variables (for dataset)
def update_variables_url(dataset_id):
    return f"{url}/{dataset_id}/update_variables"

In [4]:
# Set up the start url and payload
start_payload={"file_path": "tests/files/plate.vtp", "metadata": {"name":"plate", "unit":"m"}}

# Submit the request
resp = requests.post(start_url, json=start_payload)
resp.content
dataset_id1 = resp.json().get("dataset_id")


In [5]:
IFrame("http://localhost:8081", width="1000", height="500")

In [6]:
# List datasets and get the dataset ID
resp = requests.get(list_datasets_url)
datasets = resp.json().get('datasets', {})
datasets

{'5897798037468084': {'id': 5897798037468084,
  'name': 'plate',
  'unit': 'm',
  'file_path': 'tests/files/plate.vtp',
  'metadata_path': None}}

In [11]:
# List variables for dataset_id
resp = requests.get(list_variables_url(dataset_id1))
parts = resp.json().get('parts')
part_to_update = parts[0]
part_id = part_to_update.get("part_id")
part_name = part_to_update.get("part_name")
part_to_update

{'part_id': 5897798037468084,
 'part_name': 'plate',
 'variables': [{'index': 0,
   'type': 'POINT',
   'name': 'Normal',
   'num_components': 3,
   'num_points': 24,
   'ranges': [[0.0, 0.0], [0.0, 0.0], [0.0, 0.0]],
   'magnitude_range': [0.0, 0.0]},
  {'index': 1,
   'type': 'POINT',
   'name': 'UV',
   'num_components': 2,
   'num_points': 24,
   'ranges': [[-110.0, 110.0], [-20.0, 43.189998626708984]],
   'magnitude_range': [0.0, 118.17519190327182]},
  {'index': 0,
   'type': 'CELL',
   'name': 'Colors',
   'num_components': 3,
   'num_points': 12,
   'ranges': [[255.0, 255.0], [255.0, 255.0], [255.0, 255.0]],
   'magnitude_range': [441.6729559300637, 441.6729559300637]}]}

In [15]:
# Create the new variables for the test point dataArray (zeros):
variables = part_to_update.get("variables")

variable_to_update = variables[0]
name = variable_to_update["name"]
var_type = variable_to_update["type"]
num_points = variable_to_update["num_points"]
num_components = variable_to_update["num_components"]

print(f'updating {part_name}, variable: {name}, {var_type}, {num_components}')

new_vector_values = np.random.rand(num_points, num_components)
# need to flatten to a python list of num_points*num_components values
new_vector_values = new_vector_values.flatten().tolist()

updating plate, variable: Normal, POINT, 3


In [16]:
# Compile a dict with the vector variable metadtata + updated values
vector_update_info = {
    "type": "point",
    "name": name,
    "num_components": num_components,
    "data": new_vector_values,
    "part_id": part_id
}

In [17]:
# Run the actual update command
payload = {"variables": [vector_update_info]}
resp = requests.post(update_variables_url(dataset_id1), json=payload)